In [ ]:
!pip install pandas tabulate pymongo pandas tqdm matplotlib nltk seaborn --quiet

In [ ]:
import pymongo
from pymongo import MongoClient
import re
from tqdm import tqdm
from nltk.tokenize import word_tokenize
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
client = MongoClient("mongodb://localhost:27017")
db = client["arxiv_db"]
collection = db["papers"]

In [5]:
total_docs = collection.count_documents({})
print(f"Total dokumen dalam MongoDB: {total_docs}")

Total dokumen dalam MongoDB: 16799


In [10]:
print("\n SAMPLE DATA:")
sample_doc = collection.find_one({})
if sample_doc:
    print("Struktur dokumen:")
    for key, value in sample_doc.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"  {key}: {value[:100]}...")
        else:
            print(f"  {key}: {value}")
else:
    print(" Tidak ada data dalam collection")


 SAMPLE DATA:
Struktur dokumen:
  _id: 684a5dbd77bcceb96275fb6c
  id: http://arxiv.org/abs/2001.12004v2
  title: Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
  authors: Joseph Suarez, Yilun Du, Igor Mordatch, Phillip Isola
  summary: Progress in multiagent intelligence research is fundamentally limited by the
number and quality of e...
  published: 2020-01-31
  updated: 2020-04-17
  primary_category: cs.LG
  categories: cs.LG, cs.AI, cs.MA, stat.ML
  pdf_url: http://arxiv.org/pdf/2001.12004v2
  is_english: True
  combined_text: Neural MMO v1.3 A Massively Multiagent Game Environment for Training and Evaluating Neural Networks ...
  is_processed: True
  processed_summary: Progress in multiagent intelligence research is fundamentally limited by the number and quality of e...
  processed_title: Neural MMO v1.3 A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
  text_length: 167


In [6]:
# Statistik kategori
print("\n STATISTIK KATEGORI:")
pipeline = [
    {"$group": {"_id": "$primary_category", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
top_categories = list(collection.aggregate(pipeline))
print("Top 10 kategori:")
for cat in top_categories:
    print(f"  {cat['_id']}: {cat['count']} papers")


 STATISTIK KATEGORI:
Top 10 kategori:
  cs.CV: 2466 papers
  cs.LG: 1968 papers
  eess.IV: 1072 papers
  cs.SE: 976 papers
  cs.CR: 970 papers
  cs.DS: 854 papers
  cs.DB: 852 papers
  eess.AS: 788 papers
  eess.SY: 764 papers
  cs.GT: 741 papers


In [7]:
class MongoDBPreprocessor:
    def __init__(self, collection):
        self.collection = collection
        
    def preprocess_text(self, text):
        """Preprocessing text for BERT model"""
        if not text or pd.isna(text):
            return ""
        
        text = str(text)
        text = text.lower()
        
        # Remove URLs dan email
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove special characters
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        
        # Remove extra whitespaces
        text = ' '.join(text.split())
        
        words = text.split()
        
        return ' '.join(words)
    
    def process_sample(self, limit=5):
        """Process sample data for testing"""
        print(f" Processing {limit} sample documents...")
        
        cursor = self.collection.find({}).limit(limit)
        results = []
        
        for doc in cursor:
            original_title = doc.get('title', '')
            original_summary = doc.get('summary', '')
            
            processed_title = self.preprocess_text(original_title)
            processed_summary = self.preprocess_text(original_summary)
            combined_text = f"{processed_title} {processed_summary}".strip()
            
            result = {
                '_id': doc['_id'],
                'original_title': original_title,
                'processed_title': processed_title,
                'original_summary': original_summary[:200] + "..." if len(original_summary) > 200 else original_summary,
                'processed_summary': processed_summary[:200] + "..." if len(processed_summary) > 200 else processed_summary,
                'combined_text': combined_text[:300] + "..." if len(combined_text) > 300 else combined_text,
                'text_length': len(combined_text.split())
            }
            results.append(result)
        
        return results

# Jalankan preprocessing sample
preprocessor = MongoDBPreprocessor(collection)
sample_results = preprocessor.process_sample(limit=3)

 Processing 3 sample documents...


In [8]:
print(" HASIL PREPROCESSING SAMPLE:")
for i, result in enumerate(sample_results, 1):
    print(f"\n--- DOKUMEN {i} ---")
    print(f"ID: {result['_id']}")
    print("--------ORIGIBNAL--------")
    print(f"Original Title: {result['original_title']}")
    print(f"Original Summary: {result['original_summary']}")
    print("\n")
    print("------PROCESSED------")
    print(f"Processed Summary: {result['processed_summary']}")
    print(f"Processed Title: {result['processed_title']}")
    print("\n")
    print(f"Combined Text Length: {result['text_length']} words")
    print("-" * 50)

 HASIL PREPROCESSING SAMPLE:

--- DOKUMEN 1 ---
ID: 684a5dbd77bcceb96275fb6c
--------ORIGIBNAL--------
Original Title: Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
Original Summary: Progress in multiagent intelligence research is fundamentally limited by the
number and quality of environments available for study. In recent years,
simulated games have become a dominant research pl...


------PROCESSED------
Processed Summary: progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study in recent years simulated games have become a dominant research plat...
Processed Title: neural mmo v1 3 a massively multiagent game environment for training and evaluating neural networks


Combined Text Length: 165 words
--------------------------------------------------

--- DOKUMEN 2 ---
ID: 684a5dbd77bcceb96275fb6d
--------ORIGIBNAL--------
Original Title: Deontological Ethic

## PRE=PROCESS SELURUH DOKUMEN

In [9]:
class MongoDBPreprocessor:
    def __init__(self, collection):
        self.collection = collection
        
    def preprocess_text(self, text):
        """Preprocessing minimal untuk BERT tanpa stopword removal/lemmatisasi"""
        if not text or pd.isna(text):
            return ""
        
        text = str(text)
        
        # 1. Remove URLs dan email
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        
        # 2. Remove special characters (tapi pertahankan tanda baca dasar)
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', ' ', text)
        
        # 3. Remove extra whitespace
        text = ' '.join(text.split())
        
        return text.strip()
    
    def process_all_documents(self, batch_size=500):
        """Preprocess seluruh dokumen dan simpan ke MongoDB"""
        total_docs = self.collection.count_documents({})
        print(f"Memulai preprocessing untuk {total_docs} dokumen...")
        
        # Setup progress bar
        pbar = tqdm(total=total_docs)
        
        # Process in batches
        for i in range(0, total_docs, batch_size):
            batch = list(self.collection.find({}).skip(i).limit(batch_size))
            
            bulk_operations = []
            for doc in batch:
                # Preprocess title dan summary
                processed_title = self.preprocess_text(doc.get('title', ''))
                processed_summary = self.preprocess_text(doc.get('summary', ''))
                combined_text = f"{processed_title} {processed_summary}".strip()
                
                # Prepare update operation
                bulk_operations.append(
                    pymongo.UpdateOne(
                        {'_id': doc['_id']},
                        {'$set': {
                            'processed_title': processed_title,
                            'processed_summary': processed_summary,
                            'combined_text': combined_text,
                            'text_length': len(combined_text.split()),
                            'is_processed': True
                        }}
                    )
                )
            
            # Execute bulk write
            if bulk_operations:
                self.collection.bulk_write(bulk_operations)
            
            # Update progress bar
            pbar.update(len(batch))
        
        pbar.close()
        print(f"Preprocessing selesai. Total {total_docs} dokumen diproses.")

In [ ]:
!pip install tabulate

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017")
db = client["arxiv_db"]
collection = db["papers"]

# Inisialisasi field untuk data yang sudah diproses
collection.update_many(
    {},
    {'$set': {
        'is_processed': False,
        'processed_title': '',
        'processed_summary': '',
        'combined_text': '',
        'text_length': 0
    }},
    upsert=False
)

# Jalankan preprocessing
preprocessor = MongoDBPreprocessor(collection)
preprocessor.process_all_documents()


def display_as_dataframe(num_samples=3):
    samples = list(collection.find({'is_processed': True}).limit(num_samples))
    
    data = []
    for sample in samples:
        data.append({
            'Type': 'Original',
            'Title': sample.get('title', '')[:80] + "..." if len(sample.get('title', '')) > 80 else sample.get('title', ''),
            'Summary': sample.get('summary', '')[:100] + "..." if len(sample.get('summary', '')) > 100 else sample.get('summary', ''),
            'Length': len(sample.get('summary', '').split())
        })
        data.append({
            'Type': 'Processed', 
            'Title': sample.get('processed_title', '')[:80] + "..." if len(sample.get('processed_title', '')) > 80 else sample.get('processed_title', ''),
            'Summary': sample.get('processed_summary', '')[:100] + "..." if len(sample.get('processed_summary', '')) > 100 else sample.get('processed_summary', ''),
            'Length': sample.get('text_length', 0)
        })
    
    df = pd.DataFrame(data)
    pd.set_option('display.max_colwidth', 60)
    print(df.to_markdown(tablefmt="grid", index=False))

display_as_dataframe()

Memulai preprocessing untuk 16799 dokumen...


100%|██████████| 16799/16799 [00:03<00:00, 5476.37it/s]

Preprocessing selesai. Total 16799 dokumen diproses.
+-----------+-------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+----------+
| Type      | Title                                                                               | Summary                                                                                                 |   Length |
+===========+=====================================================================================+=========================================================================================================+==========+
| Original  | Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evalua... | Progress in multiagent intelligence research is fundamentally limited by the                            |      147 |
|           |                                                                  

In [18]:
!pip install -U sentence-transformers

  Using cached sentence_transformers-4.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached torch-2.7.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached scipy-1.15.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached typing_extensions-4.14.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached hf_xet-1.1.3-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (879 bytes)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9

In [11]:
from sentence_transformers import SentenceTransformer
from pymongo import MongoClient, UpdateOne

/home/prayatna/projects/big-data/Big-Data-Final-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ambil dokumen yang sudah diproses tapi belum punya embedding
cursor = collection.find({
    "is_processed": True,
    "embedding": {"$exists": False}
})

batch = list(cursor)
texts = [doc['combined_text'] for doc in batch]
embeddings = model.encode(texts, show_progress_bar=True)

# Simpan ke MongoDB
ops = []
for doc, embed in zip(batch, embeddings):
    ops.append(UpdateOne(
        {"_id": doc["_id"]},
        {"$set": {"embedding": embed.tolist()}}
    ))

if ops:
    collection.bulk_write(ops)
    print(f"{len(ops)} embedding berhasil disimpan.")
else:
    print("Tidak ada dokumen baru untuk diproses.")

Batches: 0it [00:00, ?it/s]

Tidak ada dokumen baru untuk diproses.


In [13]:
# Ambil beberapa dokumen yang sudah punya embedding
cursor = collection.find(
    {"embedding": {"$exists": True}},
    {
        "title": 1,
        "processed_title": 1,
        "processed_summary": 1,
        "combined_text": 1,
        "text_length": 1,
        "embedding": 1, 
        "published": 1,
        "primary_category": 1,
        "pdf_url": 1
    }
).limit(1)

# Format tampilan
docs = list(cursor)

if not docs:
    print("Belum ada dokumen dengan embedding.")
else:
    for i, doc in enumerate(docs, 1):
        print(f"\n===== DOKUMEN #{i} =====")
        print(f"Title             : {doc.get('title', '')}")
        print(f"Kategori         : {doc.get('primary_category', '')}")
        print(f"Published         : {doc.get('published', '')}")
        print(f"PDF URL           : {doc.get('pdf_url', '')}")
        print(f"Processed Title   : {doc.get('processed_title', '')}")
        print(f"Processed Summary : {doc.get('processed_summary', '')[:200]}...")
        print(f"Combined Text     : {doc.get('combined_text', '')[:200]}...")
        print(f"Text Length       : {doc.get('text_length', 0)} words")
        print(f"Embedding Sample  : {doc.get('embedding', [])} ... (dimensi: {len(doc.get('embedding', []))} total)")
        print("=" * 70)




===== DOKUMEN #1 =====
Title             : Neural MMO v1.3: A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
Kategori         : cs.LG
Published         : 2020-01-31
PDF URL           : http://arxiv.org/pdf/2001.12004v2
Processed Title   : Neural MMO v1.3 A Massively Multiagent Game Environment for Training and Evaluating Neural Networks
Processed Summary : Progress in multiagent intelligence research is fundamentally limited by the number and quality of environments available for study. In recent years, simulated games have become a dominant research pl...
Combined Text     : Neural MMO v1.3 A Massively Multiagent Game Environment for Training and Evaluating Neural Networks Progress in multiagent intelligence research is fundamentally limited by the number and quality of e...
Text Length       : 167 words
Embedding Sample  : [-0.02622826024889946, -0.059079404920339584, -0.013319289311766624, -0.02158994786441326, 0.011467953212559223, 0.049206074327

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_articles(query, top_k=5):
    # Encode query ke embedding
    query_embedding = model.encode([query])[0]
    
    # Ambil semua dokumen yang sudah punya embedding
    cursor = collection.find({"embedding": {"$exists": True}})
    
    results = []
    for doc in cursor:
        doc_embedding = np.array(doc["embedding"])
        similarity = cosine_similarity([query_embedding], [doc_embedding])[0][0]
        results.append((doc, similarity))
    
    # Urutkan berdasarkan similarity
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_k]
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n== Artikel #{i} ==")
        print(f"Title      : {doc['title']}")
        print(f"Score      : {score:.4f}")
        print(f"Authors    : {doc['authors']}")
        print(f"Published  : {doc['published']}")
        print(f"Category   : {doc['primary_category']}")
        print(f"PDF URL    : {doc['pdf_url']}")
        print(f"Summary    : {doc['summary'][:300]}...\n")

# Contoh penggunaan
recommend_articles("Convolutional Neural Networks for Image Classification")


== Artikel #1 ==
Title      : Convolutional Neural Networks as a Model of the Visual System: Past, Present, and Future
Score      : 0.5899
Authors    : Grace W. Lindsay
Published  : 2020-01-20
Category   : q-bio.NC
PDF URL    : http://arxiv.org/pdf/2001.07092v2
Summary    : Convolutional neural networks (CNNs) were inspired by early findings in the
study of biological vision. They have since become successful tools in computer
vision and state-of-the-art models of both neural activity and behavior on
visual tasks. This review highlights what, in the context of CNNs, it...


== Artikel #2 ==
Title      : Research Progress of Convolutional Neural Network and its Application in Object Detection
Score      : 0.5846
Authors    : Wei Zhang, Zuoxiang Zeng
Published  : 2020-07-27
Category   : cs.CV
PDF URL    : http://arxiv.org/pdf/2007.13284v1
Summary    : With the improvement of computer performance and the increase of data volume,
the object detection based on convolutional neural network 

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
import numpy as np

def build_lsi_model(docs, n_components=13):
    texts = [doc['combined_text'] for doc in docs]

    # TF-IDF vectorization
    vectorizer = TfidfVectorizer(stop_words='english')
    X_tfidf = vectorizer.fit_transform(texts)

    # LSI with Truncated SVD
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    normalizer = Normalizer(copy=False)
    lsi = make_pipeline(svd, normalizer)

    X_lsi = lsi.fit_transform(X_tfidf)

    return {
        "lsi_model": lsi,
        "tfidf_vectorizer": vectorizer,
        "doc_vectors": X_lsi,
        "original_docs": docs
    }


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def search_lsi(query, model_data, top_n=5):
    lsi = model_data['lsi_model']
    vectorizer = model_data['tfidf_vectorizer']
    doc_vectors = model_data['doc_vectors']
    docs = model_data['original_docs']

    query_vec = vectorizer.transform([query])
    query_lsi = lsi.transform(query_vec)

    similarities = cosine_similarity(query_lsi, doc_vectors)[0]
    top_indices = similarities.argsort()[::-1][:top_n]

    results = []
    for idx in top_indices:
        results.append({
            "title": docs[idx]["title"],
            "similarity": similarities[idx],
            "published": docs[idx].get("published", ""),
            "pdf_url": docs[idx].get("pdf_url", ""),
        })

    return results

In [26]:
cursor = collection.find({"combined_text": {"$exists": True}})
docs = list(cursor)

if not docs:
    print("Tidak ada dokumen")
else:
    lsi_data = build_lsi_model(docs, n_components=100)

    query = "machine learning for beginners"
    results = search_lsi(query, lsi_data, top_n=5)

    for i, result in enumerate(results, 1):
        print(f"\nHasil #{i}")
        print(f"Judul     : {result['title']}")
        print(f"Similarity: {result['similarity']:.4f}")
        print(f"Published : {result['published']}")
        print(f"PDF URL   : {result['pdf_url']}")



Hasil #1
Judul     : Causality Learning: A New Perspective for Interpretable Machine Learning
Similarity: 0.8180
Published : 2020-06-27
PDF URL   : http://arxiv.org/pdf/2006.16789v2

Hasil #2
Judul     : The Mathematical Foundations of Manifold Learning
Similarity: 0.8044
Published : 2020-10-30
PDF URL   : http://arxiv.org/pdf/2011.01307v1

Hasil #3
Judul     : The Mathematical Foundations of Manifold Learning
Similarity: 0.8044
Published : 2020-10-30
PDF URL   : http://arxiv.org/pdf/2011.01307v1

Hasil #4
Judul     : Learning what they think vs. learning what they do: The micro-foundations of vicarious learning
Similarity: 0.7941
Published : 2020-07-30
PDF URL   : http://arxiv.org/pdf/2007.15264v2

Hasil #5
Judul     : Augmented Q Imitation Learning (AQIL)
Similarity: 0.7915
Published : 2020-03-31
PDF URL   : http://arxiv.org/pdf/2004.00993v2
